# BIST Walk-Forward Strateji Yarışması v2 — Kapsamlı Analiz
## 10 Strateji · 40+ Fold · İstatistiksel Doğrulama · Derin Analiz

### Walk-Forward Optimizasyon (WFO) — Metodoloji

```
Expanding Pencere — Her fold sadece GEÇMİŞ veriyi kullanır:

Fold  1: [████ TRAIN (504g) ████]  [63g TEST] ▶ AL/SAT
Fold  2: [██████ TRAIN (567g) ██████]  [63g TEST] ▶ AL/SAT
Fold  3: [████████ TRAIN (630g) ████████]  [63g TEST] ▶ AL/SAT
...
Fold 40: [███████████████████ TRAIN ███████████████████]  [63g TEST] ▶ AL/SAT
                                                  ↑
                              Model bu günü GÖRMEDEN tahmin ediyor
```

---
### 10 Strateji Kataloğu

| # | Strateji | Kategori | İlham |
|---|---|---|---|
| A | Klasik TA (RSI + MACD) | Kural tabanlı | Teknik analiz |
| B | GARCH(1,1) | İstatistiksel | Akademik finans |
| C | XGBoost | Makine öğrenmesi | Gradient boosting |
| D | HMM Rejim | İstatistiksel ML | Jim Simons (Renaissance) |
| E | Kalman Filtresi | Durum-uzay modeli | Renaissance Tech |
| F | Turtle / Donchian | Trend takip CTA | Richard Dennis |
| G | Bollinger MR | Mean reversion | Quant fonları |
| H | Adaptive Momentum | Hybrid | Medallion ilhamlı |
| I | Random Forest | Makine öğrenmesi | Ensemble learning |
| J | Ensemble (çoğunluk oyu) | Kombinasyon | Portföy teorisi |

In [ ]:
import subprocess, sys

def _pip(pkg):
    print(f"  {pkg}...", end=" ", flush=True)
    r = subprocess.run([sys.executable,"-m","pip","install","-q",pkg],
                       capture_output=True, text=True)
    print("OK" if r.returncode==0 else f"HATA {r.stderr[-80:]}")

for pkg in ["pandas","numpy","scipy","scikit-learn","xgboost","arch","hmmlearn","matplotlib"]:
    _pip(pkg)

print("\nKütüphaneler hazır.")

In [ ]:
import warnings, abc, time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from scipy import stats
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from arch import arch_model

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:.3f}")
np.random.seed(42)

# ── Mod Seçimi ───────────────────────────────────────────────────────────────
# FAST_MODE = True  → Hızlı test (~5 dk)   : 1500g, ~19 fold
# FAST_MODE = False → Kapsamlı test (~15 dk): 3000g, ~40 fold
FAST_MODE = False

N_DAYS       = 1500 if FAST_MODE else 3000
WF_TRAIN_MIN = 252  if FAST_MODE else 504
WF_TEST_DAYS = 63
WF_STEP      = 63

# ── Teknik Parametreler ──────────────────────────────────────────────────────
RSI_P = 14; EMA_F, EMA_S, EMA_SIG = 12, 26, 9; VOL_W = 20
GARCH_LQ, GARCH_HQ, MOM_W = 0.35, 0.70, 5
TURTLE_ENTRY, TURTLE_EXIT   = 20, 10   # Donchian kanalı günleri
BB_WINDOW, BB_ENTRY_Z       = 20, 1.5  # Bollinger mean-reversion

# ML özellikleri
FEATURES = ["RSI","MACD","MACD_hist","MACD_signal","Vol_ratio","Rel_strength",
            "ret_1d","ret_5d","ret_20d","ATR_pct","BB_zscore","price_pos"]

COLORS = {
    "A_KlasikTA"  :"#2196F3", "B_GARCH"   :"#FF9800", "C_XGBoost":"#4CAF50",
    "D_HMM"       :"#9C27B0", "E_Kalman"  :"#00BCD4", "F_Turtle" :"#795548",
    "G_BollingerMR":"#E91E63","H_AdaptMom":"#FF5722", "I_RandForest":"#607D8B",
    "J_Ensemble"  :"#F44336", "BuyHold"   :"#9E9E9E",
}

print(f"Konfigürasyon yüklendi — {'FAST' if FAST_MODE else 'KAPSAMLI'} mod")
print(f"  Veri: {N_DAYS}g | Train min: {WF_TRAIN_MIN}g | Test: {WF_TEST_DAYS}g | Step: {WF_STEP}g")
est = (N_DAYS - WF_TRAIN_MIN) // WF_STEP
print(f"  Tahmini fold sayısı: ~{est}")

## Bölüm 1 — Özellik Mühendisliği (Genişletilmiş)

In [ ]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Genişletilmiş özellik seti: RSI, MACD, Hacim, Göreceli Güç,
    ATR, Bollinger Z-skoru, Fiyat Pozisyonu, Lag Getiriler.
    """
    df = df.copy().sort_index()
    c  = df["Close"]

    # ── Temel Getiriler ──────────────────────────────────────────────────────
    df["log_ret"] = np.log(c / c.shift(1))
    df["Label"]   = (df["log_ret"].shift(-1) > 0).astype(float)

    # ── RSI(14) ──────────────────────────────────────────────────────────────
    d = c.diff()
    df["RSI"] = 100 - (100 / (1 + d.where(d>0,0).rolling(RSI_P).mean() /
                               ((-d).where(d<0,0).rolling(RSI_P).mean() + 1e-9)))

    # ── MACD(12,26,9) ────────────────────────────────────────────────────────
    df["MACD"]        = c.ewm(span=EMA_F,adjust=False).mean() - c.ewm(span=EMA_S,adjust=False).mean()
    df["MACD_signal"] = df["MACD"].ewm(span=EMA_SIG,adjust=False).mean()
    df["MACD_hist"]   = df["MACD"] - df["MACD_signal"]

    # ── Hacim Oranı & Göreceli Güç ───────────────────────────────────────────
    df["Vol_ratio"]    = df["Volume"] / (df["Volume"].rolling(VOL_W).mean() + 1e-9)
    df["Rel_strength"] = c / (df["Endeks_Close"] + 1e-9)

    # ── Lag Getiriler ────────────────────────────────────────────────────────
    df["ret_1d"]  = df["log_ret"]
    df["ret_5d"]  = np.log(c / c.shift(5))
    df["ret_20d"] = np.log(c / c.shift(20))

    # ── ATR (Average True Range %) ───────────────────────────────────────────
    tr = pd.concat([df["High"]-df["Low"],
                    (df["High"]-c.shift(1)).abs(),
                    (df["Low"] -c.shift(1)).abs()], axis=1).max(axis=1)
    df["ATR_pct"] = tr.rolling(14).mean() / c

    # ── Bollinger Z-score ────────────────────────────────────────────────────
    bb_m = c.rolling(BB_WINDOW).mean()
    bb_s = c.rolling(BB_WINDOW).std() + 1e-9
    df["BB_zscore"] = (c - bb_m) / bb_s

    # ── Fiyat Pozisyonu (0-1 arası: 20g yüksek-düşük range'de nerede?) ───────
    h20 = df["High"].rolling(20).max()
    l20 = df["Low"].rolling(20).min()
    df["price_pos"] = (c - l20) / (h20 - l20 + 1e-9)

    # ── Donchian Kanalları (Turtle için) ─────────────────────────────────────
    df["don_high_20"] = df["High"].rolling(TURTLE_ENTRY).max()
    df["don_low_10"]  = df["Low"].rolling(TURTLE_EXIT).min()

    df.dropna(inplace=True)
    print(f"  Özellik mühendisliği: {len(df)} satır, {len(FEATURES)} ML özelliği + teknik")
    return df

## Bölüm 2 — Strateji Sınıfları

In [ ]:
# ── Temel sınıf ──────────────────────────────────────────────────────────────
class BaseStrategy(abc.ABC):
    @abc.abstractmethod
    def fit(self, df_train: pd.DataFrame) -> None: ...
    @abc.abstractmethod
    def predict(self, df_test: pd.DataFrame) -> pd.Series: ...


# ════════════════════════════════════════════════════════════════════
# A: KLASİK TA — RSI + MACD Kesişim (Stateless kural)
# ════════════════════════════════════════════════════════════════════
class ClassicTAStrategy(BaseStrategy):
    """
    MACD histogramı pozitif VE RSI 30-72 bandında → Long.
    Parametre gerektirmez; her gün bağımsız değerlendirilir.
    """
    name = "A_KlasikTA"
    def fit(self, df): pass
    def predict(self, df):
        sig = ((df["MACD_hist"] > 0) & (df["RSI"] > 30) & (df["RSI"] < 72)).astype(int)
        return sig.rename("pos")


# ════════════════════════════════════════════════════════════════════
# B: GARCH(1,1) — Oynaklık Rejimi
# ════════════════════════════════════════════════════════════════════
class GARCHStrategy(BaseStrategy):
    """
    fit : GARCH(1,1) parametrelerini train verisinden tahmin eder.
          Koşullu volatilite eşiklerini (alt %35, üst %30) hesaplar.
    predict : GARCH rekürsiyon formülü ile test volatilitesini tahmin eder.
              Düşük vol + pozitif momentum → Long; yüksek vol → Flat.
    """
    name = "B_GARCH"
    def __init__(self):
        self._w=0.05; self._a=0.05; self._b=0.90
        self._lo=None; self._hi=None; self._h=1.0; self._e2=0.01

    def fit(self, df):
        rp = (df["log_ret"] * 100).dropna()
        try:
            res = arch_model(rp, vol="Garch", p=1, q=1, mean="Zero",
                             dist="normal").fit(
                starting_values=np.array([self._w,self._a,self._b]),
                disp="off", show_warning=False, options={"maxiter":250})
            p = res.params
            self._w  = max(float(p.get("omega",0.05)),   1e-9)
            self._a  = max(float(p.get("alpha[1]",0.05)),1e-9)
            self._b  = max(float(p.get("beta[1]",0.90)), 1e-9)
            cv = res.conditional_volatility
            self._lo = float(cv.quantile(GARCH_LQ))
            self._hi = float(cv.quantile(GARCH_HQ))
            self._h  = float(cv.iloc[-1])**2
            self._e2 = float(rp.iloc[-1])**2
        except:
            ewm = (df["log_ret"]*100).ewm(span=20).std()
            self._lo = float(ewm.quantile(GARCH_LQ))
            self._hi = float(ewm.quantile(GARCH_HQ))

    def predict(self, df):
        rp  = (df["log_ret"]*100).values
        mom = df["Close"].pct_change(MOM_W).values
        h, e2 = self._h, self._e2
        vols = []
        for e in rp:
            h = self._w + self._a*e2 + self._b*h
            h = max(h, 1e-9); vols.append(np.sqrt(h)); e2=e**2
        self._h=h; self._e2=e2
        pos = []
        for v, m in zip(vols, mom):
            m = 0.0 if np.isnan(m) else m
            if v > self._hi:               pos.append(0)
            elif v < self._lo and m > 0:   pos.append(1)
            else:                          pos.append(1 if m>0 else 0)
        return pd.Series(pos, index=df.index, dtype=int, name="pos")


# ════════════════════════════════════════════════════════════════════
# C: XGBoost — Gradient Boosted Sınıflandırıcı
# ════════════════════════════════════════════════════════════════════
class XGBoostStrategy(BaseStrategy):
    """
    12 teknik özellik üzerinde XGBClassifier eğitir.
    Walk-forward fold başına ayrı model → veri sızıntısı yok.
    """
    name = "C_XGBoost"
    def __init__(self): self.model=None; self.accs=[]

    def fit(self, df):
        v = df.dropna(subset=FEATURES+["Label"])
        self.model = XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
            eval_metric="logloss", random_state=42, verbosity=0)
        self.model.fit(v[FEATURES].values, v["Label"].astype(int).values)

    def predict(self, df):
        if self.model is None:
            return pd.Series(0, index=df.index, dtype=int, name="pos")
        v     = df.dropna(subset=FEATURES)
        preds = self.model.predict(v[FEATURES].values)
        pos   = pd.Series(0, index=df.index, dtype=int, name="pos")
        pos.loc[v.index] = preds
        if "Label" in df.columns:
            self.accs.append(accuracy_score(
                df["Label"].loc[v.index].astype(int), preds))
        return pos


# ════════════════════════════════════════════════════════════════════
# D: HMM — Gizli Markov Rejim Tespiti (Jim Simons tarzı)
# ════════════════════════════════════════════════════════════════════
class HMMStrategy(BaseStrategy):
    """
    GaussianHMM ile Bull/Bear/Sideways rejim tespiti.
    Özellikler: log getiri, kısa vadeli vol, hacim oranı, göreceli güç.
    En yüksek ortalama getirili durum → Bull → Long.
    """
    name = "D_HMM"
    def __init__(self, n=3): self.n=n; self.model=None; self.bull=None

    def _feats(self, df):
        r   = df["log_ret"].values
        v5  = pd.Series(r).rolling(5).std().bfill().values
        vr  = df["Vol_ratio"].fillna(1).values
        rs  = df["Rel_strength"].fillna(df["Rel_strength"].mean()).values
        return np.column_stack([r, v5, vr, rs])

    def fit(self, df):
        from hmmlearn.hmm import GaussianHMM
        X   = self._feats(df)
        self._mu = X.mean(0); self._sg = X.std(0)+1e-9
        Xs  = (X-self._mu)/self._sg
        self.model = GaussianHMM(
            n_components=self.n, covariance_type="full",
            n_iter=200, random_state=42)
        self.model.fit(Xs)
        st  = self.model.predict(Xs)
        ret = df["log_ret"].values
        sr  = {s: ret[st==s].mean() if (st==s).sum()>0 else -999
               for s in range(self.n)}
        self.bull = max(sr, key=sr.get)

    def predict(self, df):
        try:
            X  = self._feats(df)
            Xs = (X-self._mu)/self._sg
            st = self.model.predict(Xs)
            return pd.Series((st==self.bull).astype(int), index=df.index, name="pos")
        except:
            return pd.Series(0, index=df.index, dtype=int, name="pos")


print("Stratejiler A–D tanımlandı.")

In [ ]:
# ════════════════════════════════════════════════════════════════════
# E: KALMAN FİLTRESİ — Trend Takip (Renaissance Technologies tarzı)
# ════════════════════════════════════════════════════════════════════
class KalmanStrategy(BaseStrategy):
    """
    Doğrusal Kalman filtresi ile fiyat trend tahmini.
    Durum vektörü: [log_fiyat_seviyesi, trend]
    Kalman kazancı gürültüyü filtreler; trend > 0 ise Long.

    Renaissance Technologies bu tür filtreler kullanır:
    sinyal/gürültü ayrımını olasılıksal modelleme ile yapar.
    """
    name = "E_Kalman"
    def __init__(self, q=1e-4, r=1e-2):
        self.Q = np.eye(2)*q; self.R = np.array([[r]])
        self.F = np.array([[1.,1.],[0.,1.]])
        self.H = np.array([[1.,0.]])

    def _run(self, prices, x0, P0):
        """Kalman filtresi rekürsiyon döngüsü."""
        x=x0.copy(); P=P0.copy(); trends=[]
        for p in prices:
            x=self.F@x;       P=self.F@P@self.F.T+self.Q
            y=np.array([p])-self.H@x
            S=self.H@P@self.H.T+self.R
            K=P@self.H.T@np.linalg.inv(S)
            x=x+K.flatten()*y.flatten()[0]
            P=(np.eye(2)-K@self.H)@P
            trends.append(x[1])
        return x, P, np.array(trends)

    def fit(self, df):
        lp = np.log(df["Close"].values)
        x0 = np.array([lp[0], 0.0])
        P0 = np.eye(2)*0.1
        self._x, self._P, _ = self._run(lp, x0, P0)

    def predict(self, df):
        lp = np.log(df["Close"].values)
        x, P, trends = self._run(lp, self._x, self._P)
        self._x=x; self._P=P
        pos = (trends > 0).astype(int)
        return pd.Series(pos, index=df.index, name="pos")


# ════════════════════════════════════════════════════════════════════
# F: TURTLE / DONCHIAN — CTA Klasiği (Richard Dennis & Bill Eckhardt)
# ════════════════════════════════════════════════════════════════════
class TurtleStrategy(BaseStrategy):
    """
    Donchian kanal kırılım stratejisi (Turtle Trading kuralları).
    Giriş : Close > 20g yüksek (yeni zirve → trend başlıyor)
    Çıkış : Close < 10g düşük  (trend kırıldı)
    """
    name = "F_Turtle"
    def __init__(self): self._in_pos=False

    def fit(self, df):
        # Son barı hatırla (fold başında pozisyon durumu için)
        self._in_pos = False

    def predict(self, df):
        pos = np.zeros(len(df), dtype=int)
        for i in range(len(df)):
            hi20 = df["don_high_20"].iloc[i]
            lo10 = df["don_low_10"].iloc[i]
            c    = df["Close"].iloc[i]
            if pd.isna(hi20) or pd.isna(lo10):
                pos[i] = int(self._in_pos); continue
            if not self._in_pos and c >= hi20:
                self._in_pos = True
            elif self._in_pos and c <= lo10:
                self._in_pos = False
            pos[i] = int(self._in_pos)
        return pd.Series(pos, index=df.index, name="pos")


# ════════════════════════════════════════════════════════════════════
# G: BOLLINGER MEAN REVERSION — Kantitatif Quant Klasiği
# ════════════════════════════════════════════════════════════════════
class BollingerMRStrategy(BaseStrategy):
    """
    Bollinger Bandı tabanlı ortalamaya dönüş stratejisi.
    Giriş : BB_zscore < -1.5 (aşırı satım bölgesi)
    Çıkış : BB_zscore > -0.3 (ortalamaya döndü)

    Piyasa momentum yerine mean-reversion gösterdiğinde güçlü.
    """
    name = "G_BollingerMR"
    def __init__(self): self._in_pos=False; self._entry_thr=-BB_ENTRY_Z; self._exit_thr=-0.3

    def fit(self, df): self._in_pos=False

    def predict(self, df):
        zs  = df["BB_zscore"].values
        pos = np.zeros(len(df), dtype=int)
        for i, z in enumerate(zs):
            if np.isnan(z):
                pos[i]=int(self._in_pos); continue
            if not self._in_pos and z < self._entry_thr:
                self._in_pos = True
            elif self._in_pos and z > self._exit_thr:
                self._in_pos = False
            pos[i] = int(self._in_pos)
        return pd.Series(pos, index=df.index, name="pos")


# ════════════════════════════════════════════════════════════════════
# H: ADAPTİF MOMENTUM — Medallion İlhamlı Hybrid
# ════════════════════════════════════════════════════════════════════
class AdaptiveMomStrategy(BaseStrategy):
    """
    Volatilite rejimine uyum sağlayan çok-zaman-çerçeveli momentum.
    Düşük vol → momentum sinyalleri güçlenir (trend izleme modu).
    Yüksek vol → daha kısa zaman çerçevesi, daha sık pozisyon değişimi.
    Saçılma: 5g, 10g, 20g momentumun ağırlıklı kombinasyonu.

    Medallion Fonu: farklı zaman dilimlerindeki sinyalleri birleştirerek
    sürekli pozitif getiri elde etti (1988-2018 arası yıllık %66 brüt).
    """
    name = "H_AdaptMom"
    def __init__(self): self._lo=None; self._hi=None

    def fit(self, df):
        # Train setinden vol eşiklerini hesapla
        rolling_vol = df["log_ret"].rolling(20).std()
        self._lo    = float(rolling_vol.quantile(0.33))
        self._hi    = float(rolling_vol.quantile(0.67))

    def predict(self, df):
        rv   = df["log_ret"].rolling(10).std().values
        r5   = df["ret_5d"].values
        r10  = np.log(df["Close"]/df["Close"].shift(10)).values
        r20  = df["ret_20d"].values
        rsi  = df["RSI"].values
        pos  = np.zeros(len(df), dtype=int)

        for i in range(len(df)):
            v = rv[i] if not np.isnan(rv[i]) else self._hi
            m5, m10, m20 = (r5[i] if not np.isnan(r5[i]) else 0,
                            r10[i] if not np.isnan(r10[i]) else 0,
                            r20[i] if not np.isnan(r20[i]) else 0)

            if v < self._lo:
                # Sakin piyasa: uzun vadeli momentum ağırlıklı
                score = 0.2*m5 + 0.3*m10 + 0.5*m20
            elif v > self._hi:
                # Türbülanslı piyasa: kısa vadeli, RSI filtresi güçlü
                score = 0.6*m5 + 0.3*m10 + 0.1*m20
                if not np.isnan(rsi[i]) and rsi[i] > 70:
                    score = -1   # Aşırı alımda girme
            else:
                score = 0.4*m5 + 0.4*m10 + 0.2*m20

            pos[i] = 1 if score > 0 else 0
        return pd.Series(pos, index=df.index, name="pos")


# ════════════════════════════════════════════════════════════════════
# I: RANDOM FOREST — Ensemble Makine Öğrenmesi
# ════════════════════════════════════════════════════════════════════
class RandomForestStrategy(BaseStrategy):
    """
    XGBoost'a alternatif tree ensemble. Farklı hata yapısı ve
    overfitting profili nedeniyle karşılaştırma için değerli.
    """
    name = "I_RandForest"
    def __init__(self): self.model=None; self.accs=[]

    def fit(self, df):
        v = df.dropna(subset=FEATURES+["Label"])
        self.model = RandomForestClassifier(
            n_estimators=200, max_depth=5, min_samples_split=20,
            max_features="sqrt", random_state=42, n_jobs=-1)
        self.model.fit(v[FEATURES].values, v["Label"].astype(int).values)

    def predict(self, df):
        if self.model is None:
            return pd.Series(0, index=df.index, dtype=int, name="pos")
        v     = df.dropna(subset=FEATURES)
        preds = self.model.predict(v[FEATURES].values)
        pos   = pd.Series(0, index=df.index, dtype=int, name="pos")
        pos.loc[v.index] = preds
        if "Label" in df.columns:
            self.accs.append(accuracy_score(
                df["Label"].loc[v.index].astype(int), preds))
        return pos


print("Stratejiler E–I tanımlandı.")

## Bölüm 3 — Walk-Forward Motoru

In [ ]:
BASE_STRATEGIES = {
    "A_KlasikTA"   : ClassicTAStrategy(),
    "B_GARCH"      : GARCHStrategy(),
    "C_XGBoost"    : XGBoostStrategy(),
    "D_HMM"        : HMMStrategy(),
    "E_Kalman"     : KalmanStrategy(),
    "F_Turtle"     : TurtleStrategy(),
    "G_BollingerMR": BollingerMRStrategy(),
    "H_AdaptMom"   : AdaptiveMomStrategy(),
    "I_RandForest" : RandomForestStrategy(),
}

def walk_forward_engine(
    df_feat: pd.DataFrame,
    base_strategies: dict,
    train_min: int = WF_TRAIN_MIN,
    test_days: int = WF_TEST_DAYS,
    step: int = WF_STEP,
) -> dict:
    """
    Expanding-window walk-forward backtest.

    NOT: Ensemble (J) base stratejilerin tahminlerinden
    hesaplanır — ekstra fit maliyeti yoktur.

    Veri sızıntısı garantisi:
      - Test dönemi hiçbir zaman fit() çağrısına girmez.
      - Her fold kendi bağımsız modelini eğitir.
    """
    n     = len(df_feat)
    names = list(base_strategies.keys()) + ["J_Ensemble"]
    positions = {nm: np.zeros(n, dtype=int) for nm in names}

    fold = 0; te = train_min; t0 = time.time()
    total_folds = (n - train_min) // step

    while te + test_days <= n:
        t1       = min(te + test_days, n)
        tr_df    = df_feat.iloc[:te]
        test_df  = df_feat.iloc[te:t1]
        fold_preds = {}

        for nm, strat in base_strategies.items():
            try:
                strat.fit(tr_df)
                p = strat.predict(test_df)
                positions[nm][te:t1]     = p.values[:t1-te]
                fold_preds[nm]           = p.values[:t1-te]
            except Exception as exc:
                fold_preds[nm] = np.zeros(t1-te, dtype=int)

        # Ensemble: çoğunluk oyu (≥5/9)
        vote_matrix = np.column_stack(list(fold_preds.values()))   # shape (T, 9)
        ensemble    = (vote_matrix.sum(axis=1) >= 5).astype(int)
        positions["J_Ensemble"][te:t1] = ensemble

        fold += 1; te += step
        elapsed = time.time()-t0
        eta     = elapsed/fold*(total_folds-fold) if fold > 0 else 0
        if fold % 5 == 0 or fold <= 3:
            print(f"  Fold {fold:3d}/{total_folds} | "
                  f"Geçen: {elapsed:.0f}s | ETA: {eta:.0f}s | "
                  f"Test: {str(df_feat.index[te-step])[:10]}",
                  flush=True)

    print(f"\n  Toplam fold: {fold} | Süre: {time.time()-t0:.1f}s")
    return {nm: pd.Series(arr, index=df_feat.index) for nm, arr in positions.items()}


print("Walk-Forward motoru ve 9 temel + 1 ensemble strateji hazır.")

In [ ]:
def calc_metrics(pos: pd.Series, log_ret: pd.Series,
               wf_start: int, n_boot: int = 600) -> dict:
    """
    WFO test dönemine ait tam performans metriği paketi.
    1G gecikme (sinyal günü sonu → ertesi açılış).
    """
    p  = pos.iloc[wf_start:].shift(1).fillna(0)
    r  = log_ret.iloc[wf_start:]
    s  = p * r                             # Strateji günlük log getirileri

    # Temel metrikler
    total  = float(np.expm1(s.sum()) * 100)
    std_   = s.std()
    sharpe = float(s.mean()/std_*np.sqrt(252)) if std_>1e-9 else 0.0
    cum    = np.exp(s.cumsum()); pk = cum.cummax()
    maxdd  = float(((cum-pk)/pk).min() * 100)
    calmar = total/abs(maxdd) if abs(maxdd)>1e-3 else 0.0
    act    = s[p>0]
    wr     = float((act>0).mean()*100) if len(act)>0 else 0.0
    n_tr   = int((p.diff().fillna(0)>0).sum())   # Alım sayısı
    expo   = float((p>0).mean()*100)             # % süre pozisyonda

    # Bootstrap Sharpe %95 CI
    rng = np.random.default_rng(42)
    arr = s.values
    bsh = [(lambda x: x.mean()/x.std()*np.sqrt(252) if x.std()>1e-9 else 0.0)
           (rng.choice(arr, size=len(arr), replace=True)) for _ in range(n_boot)]
    ci_lo, ci_hi = np.percentile(bsh, 2.5), np.percentile(bsh, 97.5)

    # t-test (günlük getiri > 0 mı?)
    t_, pv = (stats.ttest_1samp(act.values, 0) if len(act)>10 else (0,1))

    # Yıllık getiri dağılımı
    annual = {}
    for yr, grp in s.groupby(s.index.year):
        annual[yr] = round(float(np.expm1(grp.sum())*100), 2)

    return {
        "Getiri(%)": round(total,2), "Sharpe": round(sharpe,3),
        "CI_lo": round(ci_lo,3), "CI_hi": round(ci_hi,3),
        "MaxDD(%)": round(maxdd,2), "Calmar": round(float(calmar),3),
        "WinRate(%)": round(wr,2), "Expo(%)": round(expo,1),
        "İşlem": n_tr, "p-val": round(float(pv),4),
        "_sr": s, "_annual": annual,
    }


def champion_score(row):
    sc = row["Getiri(%)"]*0.40 + row["WinRate(%)"]*0.30 + row["Sharpe"]*0.30
    return sc * 0.8 if row["p-val"] > 0.10 else sc


print("Metrik fonksiyonları hazır.")

## Bölüm 4 — Walk-Forward Çalıştırma

In [ ]:
def run_full_comparison(df: pd.DataFrame):
    SEP = "═" * 75

    print(SEP)
    print("  BIST WALK-FORWARD STRATEJİ YARIŞMASI v2")
    print("  10 STRATEJİ | EXPANDING PENCERE | İSTATİSTİKSEL DOĞRULAMA")
    print(SEP)

    # Özellik mühendisliği
    print("\n[1/3] Özellik mühendisliği...")
    df_f = feature_engineering(df)
    n    = len(df_f)
    est  = (n - WF_TRAIN_MIN) // WF_STEP
    print(f"  Toplam: {n}g | Train min: {WF_TRAIN_MIN}g | "
          f"~{est} fold | Test başlangıç: {str(df_f.index[WF_TRAIN_MIN])[:10]}")

    # Walk-Forward
    print(f"\n[2/3] Walk-Forward başlatılıyor ({est} fold × ~7s ≈ ~{est*7//60+1} dk)...")
    wfo = walk_forward_engine(df_f, BASE_STRATEGIES)

    # Metrikler
    print("\n[3/3] Metrikler hesaplanıyor (Bootstrap CI, %600 iterasyon)...")
    results = {}; strat_rets = {}; annual_data = {}
    all_names = list(wfo.keys())

    for nm, pos in wfo.items():
        m = calc_metrics(pos, df_f["log_ret"], WF_TRAIN_MIN)
        strat_rets[nm]  = m.pop("_sr")
        annual_data[nm] = m.pop("_annual")
        results[nm]     = m
        sig = "✓" if m["p-val"]<0.10 else "~" if m["p-val"]<0.20 else "✗"
        print(f"  {nm:<16} G:{m['Getiri(%)']:+7.2f}%  Sh:{m['Sharpe']:6.3f}  "
              f"WR:{m['WinRate(%)']:5.1f}%  DD:{m['MaxDD(%)']:6.1f}%  "
              f"p={m['p-val']:.3f}{sig}")

    # Benchmark
    bh = calc_metrics(pd.Series(1, index=df_f.index), df_f["log_ret"], WF_TRAIN_MIN)
    strat_rets["BuyHold"]  = bh.pop("_sr")
    annual_data["BuyHold"] = bh.pop("_annual")
    results["BuyHold"]     = bh

    return results, strat_rets, annual_data, df_f


# Çalıştır
results, strat_rets, annual_data, df_feat = run_full_comparison(df)

## Bölüm 5 — Sonuç Tablosu & Şampiyon Seçimi

In [ ]:
# ── Kıyaslama Tablosu ────────────────────────────────────────────────────────
SHOW_COLS = ["Getiri(%)","Sharpe","CI_lo","CI_hi","MaxDD(%)","Calmar","WinRate(%)","Expo(%)","İşlem","p-val"]
df_res = pd.DataFrame(results).T[SHOW_COLS]
df_res.index.name = "Strateji"

# Sütun başlıklarını kısalt
df_display = df_res.rename(columns={
    "Getiri(%)":"Getiri%","MaxDD(%)":"MaxDD%","WinRate(%)":"WinRate%",
    "Expo(%)":"Expo%","CI_lo":"CI_düşük","CI_hi":"CI_yüksek"
})

n_test = len(df_feat) - WF_TRAIN_MIN
SEP = "═" * 75
print(SEP)
print(f"  PERFORMANS TABLOSU — WFO Out-of-Sample ({n_test} gün)")
print(f"  Buy & Hold: {results['BuyHold']['Getiri(%)']:+.2f}%  |  "
      f"Sharpe: {results['BuyHold']['Sharpe']:.3f}")
print("  ✓ p<0.10  ~ p<0.20  ✗ p>0.20  (istatistiksel anlamlılık)")
print(SEP)

# Stratejileri getiriye göre sırala
strats_sorted = sorted([k for k in results if k!="BuyHold"],
                        key=lambda x: results[x]["Getiri(%)"], reverse=True)
print(df_display.loc[strats_sorted + ["BuyHold"]].to_string())

# ── Şampiyon ─────────────────────────────────────────────────────────────────
scores     = {nm: champion_score(pd.Series(results[nm]))
              for nm in results if nm != "BuyHold"}
champion   = max(scores, key=scores.get)
runner_up  = sorted(scores, key=scores.get, reverse=True)[1]
cr         = results[champion]
bhr        = results["BuyHold"]

print("\n" + SEP)
print(f"\n  ★ ŞAMPIYON     : {champion}")
print(f"  ★ İKİNCİ       : {runner_up}")
print()
print(f"  Getiri    : {cr['Getiri(%)']:+.2f}%  (B&H: {bhr['Getiri(%)']:+.2f}%,  "
      f"alfa: {cr['Getiri(%)']-bhr['Getiri(%)']:+.2f}%)")
print(f"  Sharpe    :  {cr['Sharpe']:.3f}  CI: [{cr['CI_lo']:.3f}, {cr['CI_hi']:.3f}]")
print(f"  MaxDD     : {cr['MaxDD(%)']:.2f}%  |  Calmar: {cr['Calmar']:.3f}")
print(f"  WinRate   :  {cr['WinRate(%)']:.1f}%")
print(f"  p-value   :  {cr['p-val']:.4f}  "
      f"({'İstatistiksel olarak anlamlı ✓' if cr['p-val']<0.10 else 'Dikkat: zayıf kanıt ✗'})")
print(SEP)

# XGBoost ve RF ortalama doğrulukları
for nm, strat in BASE_STRATEGIES.items():
    if hasattr(strat, "accs") and strat.accs:
        print(f"  {nm} ort. fold doğruluğu: %{np.mean(strat.accs)*100:.2f}")

## Bölüm 6 — Derin Analiz

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# ANALİZ 1: Yıllık Getiri Kırılımı — hangi yıl hangi strateji güçlü?
# ═══════════════════════════════════════════════════════════════════
print("\n" + "─"*65)
print("ANALİZ 1 — Yıllık Getiri Kırılımı (%)")
print("─"*65)

all_years = sorted(set(yr for d in annual_data.values() for yr in d))
strats_top = [champion, runner_up] + [
    nm for nm in strats_sorted if nm not in [champion, runner_up]][:4]
strats_top = list(dict.fromkeys(strats_top))  # Tekrarı kaldır

df_annual = pd.DataFrame(
    {nm: {yr: annual_data[nm].get(yr, 0) for yr in all_years}
     for nm in strats_top + ["BuyHold"]}).T
df_annual.index.name = "Strateji"

# En iyi yıl işareti
for yr in all_years:
    best = df_annual[yr].drop("BuyHold").idxmax()
    df_annual.loc[best, yr] = str(df_annual.loc[best, yr]) + "★"

print(df_annual.to_string())

# ═══════════════════════════════════════════════════════════════════
# ANALİZ 2: Strateji Korelasyon Matrisi — hangileri çakışıyor?
# ═══════════════════════════════════════════════════════════════════
print("\n" + "─"*65)
print("ANALİZ 2 — Strateji Getiri Korelasyonu (WFO test dönemi)")
print("─"*65)

wf_start = WF_TRAIN_MIN
sr_df = pd.DataFrame({nm: strat_rets[nm].iloc[wf_start:]
                      for nm in list(strat_rets.keys()) if nm != "BuyHold"})
corr  = sr_df.corr().round(2)
print(corr.to_string())
print("\n  Korelasyon < 0.30 → Stratejiler çeşitleme sağlar (birleştirmeye değer)")
print("  Korelasyon > 0.70 → Benzer risk profili (farklı strateji ekle)")

# ═══════════════════════════════════════════════════════════════════
# ANALİZ 3: Rolling 6-Aylık Sharpe — Zaman İçinde İstikrar
# ═══════════════════════════════════════════════════════════════════
print("\n" + "─"*65)
print("ANALİZ 3 — Rolling 6-Aylık Sharpe İstikrarı")
print("─"*65)

roll_w = 126  # ~6 ay
top5 = strats_sorted[:5]
for nm in top5:
    sr     = strat_rets[nm].iloc[wf_start:]
    r_sh   = sr.rolling(roll_w).apply(
        lambda x: x.mean()/x.std()*np.sqrt(252) if x.std()>1e-9 else 0, raw=True)
    pos_pct= float((r_sh.dropna() > 0).mean() * 100)
    avg    = float(r_sh.dropna().mean())
    std_sh = float(r_sh.dropna().std())
    print(f"  {nm:<16} Ort Sharpe: {avg:+.3f}  Std: {std_sh:.3f}  "
          f"Pozitif dönem: {pos_pct:.1f}%")

# ═══════════════════════════════════════════════════════════════════
# ANALİZ 4: Rejim Performansı — Bull/Bear/Sideways
# ═══════════════════════════════════════════════════════════════════
print("\n" + "─"*65)
print("ANALİZ 4 — Piyasa Rejimi Bazında Performans")
print("─"*65)

bh_sr    = strat_rets["BuyHold"].iloc[wf_start:]
roll_bh  = bh_sr.rolling(60).sum()

# Basit rejim tespiti: 60g kümülatif B&H getirisine göre
bull_mask = roll_bh > 0.05
bear_mask = roll_bh < -0.05
side_mask = ~bull_mask & ~bear_mask

regimes = {"Bull": bull_mask, "Bear": bear_mask, "Sideways": side_mask}
print(f"  {'Strateji':<16} {'Bull':>8} {'Bear':>8} {'Sideways':>10}")
print(f"  {'─'*50}")
for nm in strats_sorted[:6] + ["BuyHold"]:
    sr = strat_rets[nm].iloc[wf_start:]
    row = []
    for mask in regimes.values():
        sub = sr[mask]
        g   = float(np.expm1(sub.sum())*100) if len(sub)>0 else 0
        row.append(f"{g:+6.1f}%")
    print(f"  {nm:<16} {row[0]:>8} {row[1]:>8} {row[2]:>10}")

print("\n" + "─"*65)
print("ANALİZ ÖZET")
print("─"*65)
print(f"  Şampiyon ({champion}) benchmark'ı {results[champion]['Getiri(%)']-results['BuyHold']['Getiri(%)']:+.2f}% geride/önde.")
print(f"  En istikrarlı Sharpe CI: {min(results, key=lambda x: results[x]['CI_hi']-results[x]['CI_lo'] if x!='BuyHold' else 999)}")
print(f"  En düşük Max Drawdown  : {min((x for x in results if x!='BuyHold'), key=lambda x: results[x]['MaxDD(%)'])}")
print(f"  En yüksek Win Rate     : {max((x for x in results if x!='BuyHold'), key=lambda x: results[x]['WinRate(%)'])}")

## Bölüm 7 — Görselleştirme (6 Panel)

In [ ]:
def plot_full_analysis(results, strat_rets, annual_data, df_feat, champion, runner_up):
    wf_start = WF_TRAIN_MIN
    top5     = sorted([k for k in results if k!="BuyHold"],
                      key=lambda x: results[x]["Getiri(%)"], reverse=True)[:5]

    fig = plt.figure(figsize=(22, 16))
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.42, wspace=0.35)

    # ── Panel 1: Kümülatif Getiri (tüm stratejiler) ──────────────────────────
    ax1 = fig.add_subplot(gs[0, :2])
    for nm in top5 + ["BuyHold"]:
        sr  = strat_rets[nm]
        cum = np.exp(sr.cumsum())
        lw  = 2.5 if nm in [champion, "BuyHold"] else 1.3
        ls  = "-" if nm == champion else (":" if nm=="BuyHold" else "--")
        pct = (cum.iloc[-1]-1)*100
        ax1.plot(cum.index, cum, label=f"{'★ ' if nm==champion else ''}{nm} ({pct:+.1f}%)",
                 color=COLORS.get(nm,"gray"), lw=lw, ls=ls)
    ax1.axhline(1, color="black", lw=0.7, ls=":", alpha=0.4)
    ax1.axvline(df_feat.index[wf_start], color="red", lw=1, ls="--", alpha=0.5,
                label=f"WFO Başlangıç")
    ax1.set_title("Kümülatif Getiri — WFO Test Dönemi", fontweight="bold", fontsize=11)
    ax1.set_ylabel("Portföy (Başlangıç=1)"); ax1.legend(fontsize=7.5, ncol=2)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:.2f}"))

    # ── Panel 2: Sharpe Karşılaştırması (CI ile) ─────────────────────────────
    ax2 = fig.add_subplot(gs[0, 2])
    nms = [k for k in results if k!="BuyHold"]
    sh  = [results[n]["Sharpe"] for n in nms]
    clo = [results[n]["CI_lo"]  for n in nms]
    chi = [results[n]["CI_hi"]  for n in nms]
    x   = np.arange(len(nms))
    err = [[s-l for s,l in zip(sh,clo)], [h-s for s,h in zip(sh,chi)]]
    ax2.barh(x, sh, color=[COLORS.get(n,"gray") for n in nms], alpha=0.8, edgecolor="white")
    ax2.errorbar(sh, x, xerr=err, fmt="none", color="black", capsize=3, lw=1.2)
    ax2.axvline(0, color="black", lw=0.8); ax2.axvline(results["BuyHold"]["Sharpe"],
                color=COLORS["BuyHold"], lw=1.2, ls="--", alpha=0.7)
    ax2.set_yticks(x); ax2.set_yticklabels(nms, fontsize=7.5)
    ax2.set_title("Sharpe (Bootstrap %95 CI)", fontweight="bold", fontsize=10)
    ax2.set_xlabel("Sharpe Oranı")

    # ── Panel 3: Yıllık Getiri (gruplu çubuğu) ───────────────────────────────
    ax3 = fig.add_subplot(gs[1, :2])
    top3 = [champion, runner_up] + [n for n in top5 if n not in [champion,runner_up]][:1]
    top3_bh = top3 + ["BuyHold"]
    yrs = sorted(set(yr for d in annual_data.values() for yr in d))
    bw  = 0.2; off = np.linspace(-(len(top3_bh)-1)*bw/2, (len(top3_bh)-1)*bw/2, len(top3_bh))
    xi  = np.arange(len(yrs))
    for i, nm in enumerate(top3_bh):
        vals = [annual_data[nm].get(yr, 0) for yr in yrs]
        ax3.bar(xi+off[i], vals, bw, label=nm,
                color=COLORS.get(nm,"gray"), alpha=0.8, edgecolor="white")
    ax3.axhline(0, color="black", lw=0.8)
    ax3.set_xticks(xi); ax3.set_xticklabels(yrs, rotation=45, fontsize=8)
    ax3.set_title("Yıllık Getiri Kırılımı", fontweight="bold", fontsize=10)
    ax3.set_ylabel("Getiri (%)"); ax3.legend(fontsize=7.5)

    # ── Panel 4: Strateji Korelasyon Matrisi ─────────────────────────────────
    ax4 = fig.add_subplot(gs[1, 2])
    sr_df = pd.DataFrame({nm: strat_rets[nm].iloc[wf_start:]
                          for nm in list(strat_rets.keys()) if nm!="BuyHold"})
    corr = sr_df.corr()
    im   = ax4.imshow(corr.values, cmap="RdYlGn", vmin=-1, vmax=1)
    ax4.set_xticks(range(len(corr))); ax4.set_yticks(range(len(corr)))
    ax4.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=6)
    ax4.set_yticklabels(corr.index, fontsize=6)
    for i in range(len(corr)):
        for j in range(len(corr)):
            ax4.text(j, i, f"{corr.iloc[i,j]:.1f}", ha="center", va="center",
                     fontsize=5.5, color="black" if abs(corr.iloc[i,j])<0.7 else "white")
    plt.colorbar(im, ax=ax4, shrink=0.8)
    ax4.set_title("Getiri Korelasyonu", fontweight="bold", fontsize=10)

    # ── Panel 5: Rolling 6-Aylık Sharpe ──────────────────────────────────────
    ax5 = fig.add_subplot(gs[2, :2])
    roll_w = 126
    for nm in top5:
        sr  = strat_rets[nm].iloc[wf_start:]
        rsh = sr.rolling(roll_w).apply(
            lambda x: x.mean()/x.std()*np.sqrt(252) if x.std()>1e-9 else 0, raw=True)
        ax5.plot(rsh.index, rsh, label=f"{'★ ' if nm==champion else ''}{nm}",
                 color=COLORS.get(nm,"gray"),
                 lw=2 if nm==champion else 1.2, alpha=0.85)
    ax5.axhline(0, color="black", lw=0.8, ls="--", alpha=0.5)
    ax5.set_title("Rolling 6-Aylık Sharpe — Zaman İçinde İstikrar", fontweight="bold", fontsize=10)
    ax5.set_ylabel("Sharpe (6 aylık)"); ax5.legend(fontsize=7.5)
    ax5.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:.1f}"))

    # ── Panel 6: Getiri vs Risk scatter (MaxDD) ───────────────────────────────
    ax6 = fig.add_subplot(gs[2, 2])
    for nm in [k for k in results if k!="BuyHold"]:
        r = results[nm]
        ax6.scatter(-r["MaxDD(%)"], r["Getiri(%)"], s=100,
                    color=COLORS.get(nm,"gray"), zorder=5, edgecolors="white", lw=1.2)
        ax6.annotate(nm.split("_")[0][:8], (-r["MaxDD(%)"], r["Getiri(%)"]),
                     textcoords="offset points", xytext=(4,2), fontsize=6.5)
    ax6.axhline(results["BuyHold"]["Getiri(%)"], color=COLORS["BuyHold"],
                ls="--", lw=1, alpha=0.7)
    ax6.set_xlabel("Düşen Drawdown (MaxDD → sağ = daha iyi)"); ax6.set_ylabel("Getiri (%)")
    ax6.set_title("Getiri / Risk Dengesi", fontweight="bold", fontsize=10)

    plt.suptitle(
        f"BIST Walk-Forward Analizi ({N_DAYS}g | ~{(N_DAYS-WF_TRAIN_MIN)//WF_STEP} fold)  "
        f"|  Şampiyon: {champion}",
        fontsize=12, fontweight="bold", y=1.005)

    plt.savefig("/tmp/wfo_v2_analiz.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("Grafik: /tmp/wfo_v2_analiz.png")


plot_full_analysis(results, strat_rets, annual_data, df_feat, champion, runner_up)

## Bölüm 8 — Şampiyon Stratejinin Güncel Sinyali & Sonraki Adım

In [ ]:
def generate_final_signal(df_feat, champion_name, base_strategies):
    """
    Şampiyon stratejiyi TÜM geçmiş veriye fit eder.
    En son günün özellikleri ile SONRAKI GÜN sinyalini üretir.
    """
    SEP2 = "─" * 60
    strat = base_strategies[champion_name]

    # Son bar: label bilinmiyor; geçmiş özellikler geçerli
    valid_train = df_feat.dropna(subset=["Label"])
    last_bar    = df_feat.iloc[[-1]]

    print(f"\n{SEP2}")
    print(f"  Şampiyon ({champion_name}) tüm veriye ({len(valid_train)}g) fit ediliyor...")
    strat.fit(valid_train)
    pred = strat.predict(last_bar)
    sig  = int(pred.iloc[0])

    # Olasılık (ML modelleri için)
    prob_up = None
    if hasattr(strat, "model") and strat.model is not None:
        if hasattr(strat.model, "predict_proba"):
            try:
                pv = strat.model.predict_proba(last_bar[FEATURES].values)[0]
                prob_up = float(pv[1])
            except: pass

    last_date = str(df_feat.index[-1])[:10]
    print(f"  Son veri tarihi: {last_date}")
    print(SEP2)

    if sig == 1:
        print("  ╔══════════════════════════════════════╗")
        print("  ║   SONRAKI GÜN SİNYALİ:  AL ▲ LONG   ║")
        print("  ╚══════════════════════════════════════╝")
    else:
        print("  ╔══════════════════════════════════════╗")
        print("  ║   SONRAKI GÜN SİNYALİ:  FLAT ▬ NAKİT ║")
        print("  ╚══════════════════════════════════════╝")

    if prob_up is not None:
        bar_u = "█" * int(prob_up * 30)
        bar_d = "░" * (30 - int(prob_up * 30))
        print(f"\n  Yükseliş olasılığı: {prob_up*100:.1f}%  [{bar_u}{bar_d}]")
        print(f"  Düşüş   olasılığı: {(1-prob_up)*100:.1f}%")

    # Güncel teknik tablo
    lt = df_feat.iloc[-1]
    print(f"\n  Güncel Göstergeler ({last_date}):")
    for col, lbl in [("RSI","RSI(14)"),("MACD_hist","MACD Hist"),
                     ("BB_zscore","Bollinger Z"),("Vol_ratio","Hacim Oranı"),
                     ("price_pos","Fiyat Poz."),("ret_5d","5G Getiri"),
                     ("ret_20d","20G Getiri"),("ATR_pct","ATR %")]:
        val = lt.get(col, float("nan"))
        if col in ["ret_5d","ret_20d","ATR_pct"]: val *= 100
        print(f"    {lbl:<16}: {val:+.3f}")
    print(SEP2)

    return {"signal": sig, "prob_up": prob_up, "date": last_date}


signal_info = generate_final_signal(df_feat, champion, BASE_STRATEGIES)

# ── Sonraki Adım ─────────────────────────────────────────────────────────────
print("""
┌─────────────────────────────────────────────────────────────┐
│              SONRAKI AŞAMA ÖNERİLERİ                        │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│ 1. POZİSYON BOYUTLANDIRMA                                   │
│    Kelly Kriteri ile optimal portföy ağırlığı               │
│    Her fold'dan hesaplanan win-rate/payoff → kelly%          │
│                                                             │
│ 2. RİSK YÖNETİMİ                                            │
│    Stop-loss (2×ATR veya sabit %)                            │
│    Maksimum günlük kayıp limiti                              │
│    Portföy seviyesi VaR/CVaR                                │
│                                                             │
│ 3. ÇOK VARLIKLI PORTFÖY                                      │
│    Birden fazla hissede strateji uygulama                   │
│    Korelasyon bazlı ağırlıklandırma                         │
│    Minimum varyans portföyü                                 │
│                                                             │
│ 4. PARAMETRİ OPTİMİZASYON                                   │
│    Walk-forward parametre optimizasyonu                     │
│    Bayesian optimizasyon (over-fitting'siz)                 │
│                                                             │
│ 5. GERÇEK VERİ DOĞRULAMASI                                  │
│    yfinance ile THYAO/GARAN/ASELS testleri                  │
│    Komisyon ve likidite maliyeti eklenmesi                  │
│                                                             │
│ Bir sonraki aşamayı seçin →                                 │
└─────────────────────────────────────────────────────────────┘
""")

## Demo Verisi & Tam Çalıştırma

In [ ]:
def create_synthetic_bist(n_days=3000, seed=42):
    """
    Gerçekçi BIST hisse sentetik veri üretimi.
    GARCH(1,1) volatilite kümelenmesi + rejim geçişleri + hacim korelasyonu.
    """
    rng    = np.random.default_rng(seed)
    mu     = 7e-4   # ~yılda +19% drift (enflasyonist BIST)
    sigma0 = 0.018  # Başlangıç vol

    # GARCH benzeri volatilite serisi
    eps = rng.standard_normal(n_days)
    vol = np.zeros(n_days); vol[0] = sigma0
    for t in range(1, n_days):
        vol[t] = np.sqrt(max(1e-6,
            sigma0**2*0.04 + 0.09*(vol[t-1]*eps[t-1])**2 + 0.87*vol[t-1]**2))

    # Piyasa rejim geçişleri (bull/bear periyotları)
    regime = np.ones(n_days)
    rng_r  = np.random.default_rng(123)
    in_bear = False
    for t in range(100, n_days):
        if not in_bear and rng_r.random() < 0.002:
            in_bear = True
        elif in_bear and rng_r.random() < 0.008:
            in_bear = False
        if in_bear:
            regime[t] = -0.5

    log_r = regime * mu + vol * eps

    close  = 50 * np.exp(np.cumsum(log_r))
    intra  = 0.013
    high   = close * np.exp( abs(rng.normal(0, intra, n_days)))
    low    = close * np.exp(-abs(rng.normal(0, intra, n_days)))
    open_  = close * np.exp(rng.normal(0, intra*0.4, n_days))
    volume = (900_000*(1+abs(log_r)/sigma0)*rng.lognormal(0,0.4,n_days)).astype(int)
    idx_lr = 0.65*log_r + 0.35*(5e-4+0.013*rng.standard_normal(n_days))
    endeks = 9000*np.exp(np.cumsum(idx_lr))

    dates = pd.bdate_range("2015-01-05", periods=n_days, freq="B")
    df    = pd.DataFrame({
        "Open":open_, "High":high, "Low":low, "Close":close,
        "Volume":volume, "Endeks_Close":endeks,
    }, index=dates)
    df.index.name = "Date"

    print(f"Sentetik veri: {n_days}g | {dates[0].date()} → {dates[-1].date()}")
    print(f"  Close: {close[0]:.2f} → {close[-1]:.2f} TL  "
          f"(Kümülatif: {(close[-1]/close[0]-1)*100:+.1f}%)")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# TAM ANALİZİ ÇALIŞTIR
# ─────────────────────────────────────────────────────────────────────────────
# Önce veriyi oluştur (ya da gerçek df kullan)
df = create_synthetic_bist(n_days=N_DAYS, seed=42)

# Sonra tüm analizi çalıştır (yukarıdaki hücrelerin çalışmış olması gerekir)
# run_full_comparison(df) yukarıdaki hücrede çağrılır.
# Bu hücre sadece veriyi hazırlar — tek hücre çalıştırmak için:

# results, strat_rets, annual_data, df_feat = run_full_comparison(df)
# signal_info = generate_final_signal(df_feat, champion, BASE_STRATEGIES)